# Telco Customer Churn — Data Cleaning

Preparing the raw Telco churn dataset for analysis in MySQL and Power BI.

**Source:** IBM Telco Customer Churn dataset (7,043 customers, 21 columns)

In [ ]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

## First look at the data

Checking data types and missing values before making any changes.

In [ ]:
df.info()
print("\nMissing values:", df.isnull().sum().sum())

## Problem: TotalCharges is text, not a number

Every other charge column is numeric, but `TotalCharges` came in as text.
That means it can't be summed or averaged. Finding out why before fixing it.

In [ ]:
blanks = df["TotalCharges"].astype(str).str.strip() == ""
print("Rows with blank TotalCharges:", blanks.sum())

df.loc[blanks, ["customerID", "tenure", "MonthlyCharges", "Contract", "Churn"]]

## Decision: fill blanks with 0

All 11 blank rows have **tenure = 0** — customers who signed up but haven't
been through a billing cycle yet. These aren't data errors; they genuinely
have been charged nothing.

Setting them to 0 rather than dropping them. Two reasons:
- Zero is the literally correct value for a customer who hasn't been billed
- 11 rows out of 7,043 changes nothing, and keeping them preserves the full customer count

The alternative would be dropping them, on the grounds that a tenure-0
customer hasn't had the opportunity to churn. Both are defensible.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("TotalCharges is now:", df["TotalCharges"].dtype)
print("Missing:", df["TotalCharges"].isnull().sum())

## Duplicates and consistency

In [ ]:
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())

`SeniorCitizen` is stored as 0/1 while every other yes/no column uses text.
Converting it so Power BI reads all the categorical fields the same way.

In [ ]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

df["SeniorCitizen"].value_counts()

## Sanity check before saving

Verifying the headline numbers look right.

In [ ]:
print("Rows:", len(df))
print("Churn rate:", round((df["Churn"] == "Yes").mean() * 100, 2), "%")
print("Annual revenue at risk: $" +
      format(round(df.loc[df["Churn"] == "Yes", "MonthlyCharges"].sum() * 12, 2), ","))

In [ ]:
df.to_csv("customer_retention_analytics_data.csv", index=False)
print("Saved as customer_retention_analytics_data.csv")